In [ ]:
# Entrenamiento
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import pickle

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

models = []
for i in range(8):
    clf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=i)
    clf.fit(X_train, y_train)
    models.append(clf)

with open(r"..\samples\models.pkl", "wb") as f:
    pickle.dump(models, f)

print(f"Modelos entrenados: {len(models)}")

Modelos entrenados: 8


In [ ]:
# Sacar pesos
import pickle
import json
import numpy as np

with open(r"..\samples\models.pkl", "rb") as f:
    models = pickle.load(f)

weights = []
for model_idx, m in enumerate(models):
    for layer_idx, coef in enumerate(m.coefs_):
        weights.append({
            "model": model_idx,
            "layer": layer_idx,
            "weights": coef.tolist()
        })

with open(r"..\samples\weights.json", "w") as f:
    json.dump(weights, f)

print(f"Pesos guardados: {len(weights)} bloques")

Pesos guardados: 24 bloques


In [ ]:
# Aplicar t-SNE
from sklearn.manifold import TSNE
import numpy as np
import json

with open(r"..\samples\weights.json", "r") as f:
    weights = json.load(f)

# --- Epoch viz: neuronas de layer 0 de cada modelo (64 neuronas × 3 modelos) ---
epoch_vectors, epoch_meta = [], []
for w in weights:
    if w["layer"] == 0:
        arr = np.array(w["weights"])  # shape (64, 64)
        for j in range(arr.shape[1]):
            epoch_vectors.append(arr[:, j])
            epoch_meta.append({"model": w["model"], "neuron": j, "label": j % 10})

epoch_vectors = np.vstack(epoch_vectors)
emb_epoch = TSNE(n_components=2, random_state=0, perplexity=30).fit_transform(epoch_vectors)
print(f"t-SNE epoch: {emb_epoch.shape}")

# --- Layer viz: primeras 10 neuronas de cada capa del modelo 0 ---
N = 10  # min neuronas (capa de salida tiene 10)
layer_vectors, layer_meta = [], []
for w in weights:
    if w["model"] == 0:
        arr = np.array(w["weights"])
        n_neurons = min(arr.shape[1], N)
        for j in range(n_neurons):
            layer_vectors.append(np.pad(arr[:, j], (0, max(0, 64 - arr.shape[0]))))
            layer_meta.append({"layer": w["layer"], "neuron": j, "label": j})

layer_vectors = np.vstack(layer_vectors)
emb_layer = TSNE(n_components=2, random_state=0, perplexity=5).fit_transform(layer_vectors)
print(f"t-SNE layer: {emb_layer.shape}")

t-SNE epoch: (512, 2)
t-SNE layer: (30, 2)


In [28]:
# Celda 4: Armar los JSONs con la estructura que esperan los JS
import json
from collections import defaultdict

# --- epoch_by_epoch: { "epochs": [ { "points": [...] } × n_models ] } ---
epoch_points = [
    {**m, "x": float(e[0]), "y": float(e[1])}
    for e, m in zip(emb_epoch.tolist(), epoch_meta)
]
epochs_by_model = defaultdict(list)
for p in epoch_points:
    epochs_by_model[p["model"]].append(p)

epoch_json = {"epochs": [{"points": epochs_by_model[i]} for i in sorted(epochs_by_model)]}

with open(r"..\samples\tsne_epoch_by_epoch_fc4.json", "w") as f:
    json.dump(epoch_json, f)
print(f"epoch JSON: {len(epoch_json['epochs'])} epochs, {len(epoch_json['epochs'][0]['points'])} puntos c/u")

# --- layer_by_layer: { "layers": { "layer_0": [...], "layer_1": [...], "layer_2": [...] } }
# Todas las listas deben tener el mismo tamaño (N=10)
layer_points = [
    {**m, "x": float(e[0]), "y": float(e[1])}
    for e, m in zip(emb_layer.tolist(), layer_meta)
]
layers_by_name = defaultdict(list)
for p in layer_points:
    layers_by_name[f"layer_{p['layer']}"].append(p)

layer_json = {"layers": dict(layers_by_name)}

with open(r"..\samples\tsne_layer_by_layer.json", "w") as f:
    json.dump(layer_json, f)
print(f"layer JSON: {list({k: len(v) for k, v in layer_json['layers'].items()})}")

epoch JSON: 8 epochs, 64 puntos c/u
layer JSON: ['layer_0', 'layer_1', 'layer_2']


In [29]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


In [30]:
import json

with open(r"..\samples\tsne_epoch_by_epoch_fc4.json", "r") as f:
    epoch_data = json.load(f)

epoch_data

{'epochs': [{'points': [{'model': 0,
     'neuron': 0,
     'label': 0,
     'x': 0.3600345849990845,
     'y': -2.251288652420044},
    {'model': 0,
     'neuron': 1,
     'label': 1,
     'x': 0.37375882267951965,
     'y': -1.1552977561950684},
    {'model': 0,
     'neuron': 2,
     'label': 2,
     'x': 0.7778559923171997,
     'y': 0.6060284972190857},
    {'model': 0,
     'neuron': 3,
     'label': 3,
     'x': 0.5041734576225281,
     'y': 0.16649845242500305},
    {'model': 0,
     'neuron': 4,
     'label': 4,
     'x': 0.9146193265914917,
     'y': -0.4611717760562897},
    {'model': 0,
     'neuron': 5,
     'label': 5,
     'x': 0.7625827193260193,
     'y': -0.5679365992546082},
    {'model': 0,
     'neuron': 6,
     'label': 6,
     'x': -0.804087221622467,
     'y': 1.5980452299118042},
    {'model': 0,
     'neuron': 7,
     'label': 7,
     'x': -0.24976758658885956,
     'y': 0.28738781809806824},
    {'model': 0,
     'neuron': 8,
     'label': 8,
     'x': 0.1918

In [31]:
from vizpro import RangeInput

r = RangeInput(min_val =0,max_val = 4)
r

In [32]:
from vizpro import CustomWidget
import traitlets
class CustomEpoch(CustomWidget):
    _esm = CustomWidget.createWidgetFromLocalFile(paramList=["data"], filePath=r"..\samples\d3_epoch.js")
    
    data = traitlets.Dict({}).tag(sync=True)

plot1 = CustomEpoch(data=epoch_data)

In [33]:
plot1

In [34]:

with open(r"..\samples\tsne_layer_by_layer.json", "r") as f:
    layer_data = json.load(f)

layer_data

{'layers': {'layer_0': [{'layer': 0,
    'neuron': 0,
    'label': 0,
    'x': 41.591609954833984,
    'y': 28.25372314453125},
   {'layer': 0,
    'neuron': 1,
    'label': 1,
    'x': 40.46187973022461,
    'y': 4.463122844696045},
   {'layer': 0,
    'neuron': 2,
    'label': 2,
    'x': 12.090363502502441,
    'y': 32.84020233154297},
   {'layer': 0,
    'neuron': 3,
    'label': 3,
    'x': 3.570467948913574,
    'y': 8.941967964172363},
   {'layer': 0,
    'neuron': 4,
    'label': 4,
    'x': -15.035058975219727,
    'y': 11.729753494262695},
   {'layer': 0,
    'neuron': 5,
    'label': 5,
    'x': -3.8020009994506836,
    'y': -14.355381965637207},
   {'layer': 0,
    'neuron': 6,
    'label': 6,
    'x': -44.729896545410156,
    'y': 20.032236099243164},
   {'layer': 0,
    'neuron': 7,
    'label': 7,
    'x': 13.799148559570312,
    'y': -8.904120445251465},
   {'layer': 0,
    'neuron': 8,
    'label': 8,
    'x': 1.8566200733184814,
    'y': 29.75369644165039},
   {'layer

In [35]:
class CustomLayer(CustomWidget):
    _esm = CustomWidget.createWidgetFromLocalFile(paramList=["data"], filePath=r"..\samples\d3_layer.js")
    
    data = traitlets.Dict({}).tag(sync=True)

plot2 = CustomLayer(data=layer_data)

In [36]:
plot2